# 04e. Veto de mãe

Censo sem CPF no [`04_atribuir.ipynb`](04_atribuir.ipynb) cujo **melhor par**
cai no veto de mãe: as duas mães preenchidas, nem iguais nem prefixo
(`n ≥ 2` no começo), e não é o caso de nome completo e data iguais.

O [`04c_decompor_sem_cpf.ipynb`](04c_decompor_sem_cpf.ipynb) mistura isso com
primeiro nome ausente ou com mais de 1 de Levenshtein. Aqui os dois ficam
separados. A escada já aceita Jaro-Winkler da mãe ≥ 0,75, e de 0,95 a 0,99
não aplica esses vetos. Este notebook continua mostrando o corte estrito.


In [1]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    SPLINK_ATRIBUICAO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b antes)')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

splink_input → censo_limpo_aplicacao ∪ cpf_limpo_aplicacao


('censo_limpo_aplicacao', 'cpf_limpo_aplicacao')

In [2]:
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_2 AS
SELECT unique_id_censo
FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')

print(
    'Pares no parquet:',
    f"{con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]:,}",
)
print(
    'Atribuições:',
    f"{con.execute('SELECT COUNT(*) FROM atribuicao_2').fetchone()[0]:,}",
)


Pares no parquet: 6,598,211
Atribuições: 4,065,973


Melhor par de quem não saiu no 04. `veto_mae` exige primeiro
nome preenchido e com no máximo 1 de Levenshtein: se o primeiro nome já
reprova, o par vai para `so_primeiro` e não entra na leitura da mãe.


In [3]:
tok_c = "string_split(ca.nome_mae_phon, ' ')"
tok_p = "string_split(pb.nome_mae_phon, ' ')"
n_mae = f"least(len({tok_c}), len({tok_p}))"
prefixo_mae = (
    f"{n_mae} >= 2 AND "
    f"list_slice({tok_c}, 1, {n_mae}) = list_slice({tok_p}, 1, {n_mae})"
)
# Nome completo e data iguais passam mesmo com mãe diferente.
excecao = (
    "ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL "
    "AND ca.nome_completo_phon = pb.nome_completo_phon "
    "AND ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento = pb.data_nascimento"
)

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
      SELECT 1 FROM atribuicao_2 a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute('''
CREATE OR REPLACE TABLE melhor_par AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM splink_predictions
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY unique_id_censo
    ORDER BY match_probability DESC, unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_sem AS
SELECT
    s.unique_id AS unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    CASE
        WHEN p.unique_id_cpf IS NULL THEN 'sem_par'
        WHEN ca.primeiro_nome_phon IS NULL
             OR pb.primeiro_nome_phon IS NULL
             OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) > 1
            THEN 'so_primeiro'
        WHEN ca.nome_mae_phon IS NOT NULL
             AND pb.nome_mae_phon IS NOT NULL
             AND ca.nome_mae_phon <> pb.nome_mae_phon
             AND NOT ({prefixo_mae})
             AND NOT ({excecao})
            THEN 'veto_mae'
        ELSE 'passaria'
    END AS porta
FROM censo_sem_cpf s
LEFT JOIN melhor_par p ON p.unique_id_censo = s.unique_id
LEFT JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = s.unique_id
LEFT JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
''')

display(con.execute('''
SELECT
    porta,
    COUNT(*) AS n,
    SUM(CASE WHEN match_probability >= 0.75 THEN 1 ELSE 0 END) AS n_p075,
    ROUND(AVG(match_probability), 3) AS avg_p
FROM melhor_sem
GROUP BY 1
ORDER BY n DESC
''').df())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,porta,n,n_p075,avg_p
0,sem_par,1463222,0.0,NaN
1,passaria,457554,175891.0,0.500
2,veto_mae,75201,15933.0,0.352
3,so_primeiro,17080,12533.0,0.714


No veto de mãe, como os dois nomes da mãe se afastam, ao lado
do nome da pessoa e da data. `mesmas_palavras_ordem` e `contem_sem_prefixo`
são o lugar em que a comparação pode estar rígida demais: as palavras batem,
a regra de prefixo não.


In [4]:
tok_c = "string_split(ca.nome_mae_phon, ' ')"
tok_p = "string_split(pb.nome_mae_phon, ' ')"
n_c = f"len({tok_c})"
n_p = f"len({tok_p})"
inter = f"len(list_intersect({tok_c}, {tok_p}))"

con.execute(f'''
CREATE OR REPLACE TABLE veto_mae AS
SELECT
    m.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    ca.nome_completo_phon AS nome_censo,
    pb.nome_completo_phon AS nome_cpf,
    ca.nome_mae_phon AS mae_censo,
    pb.nome_mae_phon AS mae_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) AS jw_mae,
    CASE
        WHEN list_sort({tok_c}) = list_sort({tok_p}) AND {tok_c} <> {tok_p}
            THEN 'mesmas_palavras_ordem'
        WHEN {tok_c} = {tok_p}
            THEN 'mesmas_palavras'
        WHEN {inter} = least({n_c}, {n_p}) AND {n_c} <> {n_p}
            THEN 'contem_sem_prefixo'
        WHEN jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) >= 0.92
            THEN 'parecidas'
        WHEN {inter} >= 1
            THEN 'alguma_palavra'
        ELSE 'sem_palavra'
    END AS mae_dif,
    CASE
        WHEN ca.nome_completo_phon = pb.nome_completo_phon THEN 'exato'
        WHEN ca.primeiro_nome_phon = pb.primeiro_nome_phon
             AND ca.ultimo_nome_phon = pb.ultimo_nome_phon THEN 'pontas'
        ELSE 'outro'
    END AS nome_pessoa,
    CASE
        WHEN ca.data_nascimento IS NULL AND pb.data_nascimento IS NULL THEN 'as_duas_nulas'
        WHEN ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL THEN 'uma_nula'
        WHEN ca.data_nascimento = pb.data_nascimento THEN 'exata'
        ELSE 'diferente'
    END AS data_par
FROM melhor_sem m
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE m.porta = 'veto_mae'
''')

n_excecao = con.execute('''
SELECT COUNT(*) FROM veto_mae
WHERE nome_pessoa = 'exato' AND data_par = 'exata'
''').fetchone()[0]
print('Veto de mãe:', f"{con.execute('SELECT COUNT(*) FROM veto_mae').fetchone()[0]:,}")
print('Nome e data exatos dentro do veto (tem de ser 0):', n_excecao)

display(con.execute('''
SELECT
    mae_dif,
    nome_pessoa,
    data_par,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p,
    ROUND(AVG(jw_mae), 3) AS avg_jw_mae
FROM veto_mae
GROUP BY 1, 2, 3
ORDER BY n DESC
''').df())


Veto de mãe: 75,201
Nome e data exatos dentro do veto (tem de ser 0): 0


,mae_dif,nome_pessoa,data_par,n,pct,avg_p,avg_jw_mae
0,alguma_palavra,exato,diferente,27136,36.1,0.308,0.678
1,alguma_palavra,exato,uma_nula,14035,18.7,0.317,0.673
2,sem_palavra,exato,diferente,9008,12.0,0.378,0.594
3,sem_palavra,exato,uma_nula,6124,8.1,0.396,0.589
4,alguma_palavra,outro,exata,2786,3.7,0.796,0.665
5,sem_palavra,outro,exata,2491,3.3,0.791,0.592
6,parecidas,exato,diferente,1882,2.5,0.076,0.961
7,alguma_palavra,pontas,diferente,1682,2.2,0.161,0.692
8,alguma_palavra,pontas,exata,1663,2.2,0.726,0.666
9,sem_palavra,pontas,diferente,1650,2.2,0.151,0.605


Amostra. Primeiro as mães com as mesmas palavras ou uma dentro
da outra. Depois as parecidas com nome da pessoa exato. Por último, mãe sem
palavra em comum e nome da pessoa exato: o veto no caso claro.


In [5]:
cols = '''
    match_probability,
    mae_dif,
    nome_pessoa,
    data_par,
    ROUND(jw_mae, 3) AS jw_mae,
    nome_censo,
    nome_cpf,
    mae_censo,
    mae_cpf,
    dob_censo,
    dob_cpf
'''

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif IN ('mesmas_palavras_ordem', 'mesmas_palavras', 'contem_sem_prefixo')
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif = 'parecidas' AND nome_pessoa = 'exato'
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif = 'sem_palavra' AND nome_pessoa = 'exato'
ORDER BY match_probability DESC
LIMIT 20
''').df())


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.843428,contem_sem_prefixo,exato,diferente,0.656,KLARISI KARDOZU,KLARISI KARDOZU,EUNICE,MARIA EUNICE CARDOSO,2011-11-27,2011-09-27
1,0.799881,contem_sem_prefixo,outro,exata,0.684,MARIA ALISI RODRIGUIS,MARIA ALISI RODRIGUIS BRAGA,CASSIA RODRIGUES,LEONIA CASSIA LOPES RODRIGUES,2022-04-02,2022-04-02
2,0.798351,contem_sem_prefixo,outro,exata,0.856,TEREZA VITORIA SOARIS,TEREZA VITORIA SOARIS SOUZA KONSEIKQAU,ERICA SOARES,ERICA CRISTINA SOARES SOUZA,2013-07-08,2013-07-08
3,0.798045,contem_sem_prefixo,outro,exata,0.877,MARIA ALISI,MARIA ALISI BELU MENDIS,VANESSA BELO,VANESSA MARIA BELO SOUSA MENDES,2016-03-02,2016-03-02
4,0.796254,contem_sem_prefixo,pontas,exata,0.671,MARIA EDUARDA SIUVA,MARIA EDUARDA BATISTA SIUVA,ANA BATISTA,ANA PATRICIA AZEVEDO BATISTA,2021-09-26,2021-09-26
5,0.795914,contem_sem_prefixo,outro,exata,0.655,MARIA LIVRAMENTU,MARIA LIVRAMENTU FONSEKA SOUZA,SOCORRO,MARIA SOCORRO SANTOS FONSECA,1981-09-29,1981-09-29
6,0.793191,contem_sem_prefixo,outro,exata,0.677,ANA VIRJINIA GAIOZU,ANA VIRJINIA GAIOZU AUMENDRA AUBUKERKI,LINA GAYOSO,LINA JOSEPHINA AREA LEAO GAYOSO ALMENDRA ALBUQ...,2013-04-03,2013-04-03
7,0.793191,contem_sem_prefixo,outro,exata,0.655,RAIMUNDU SANTUS,RAIMUNDU SANTUS FONSEKA,SOCORRO,MARIA SOCORRO SANTOS FONSECA,1992-08-17,1992-08-17
8,0.792288,contem_sem_prefixo,outro,exata,0.863,ANA TERA,ANA TERA MODESTU LIMA PIRIS,SHEILA MOREIRA LIMA,SHEILA NEUZA MOREIRA LIMA MODESTO,1992-09-12,1992-09-12
9,0.790486,contem_sem_prefixo,outro,exata,0.892,ANJELA MARIA,ANJELA MARIA SANTUS SANTUS,SILVA HELENA,SILVIA HELENA SILVA SANTOS,2003-05-31,2003-05-31


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.391893,parecidas,exato,uma_nula,0.930,POLIANA SIUVA OLIVEIRA,POLIANA SIUVA OLIVEIRA,VANESSA TRAGINO SILVA,VANESSA TARGINO SILVA OLIVEIRA,None,2010-12-11
1,0.387055,parecidas,exato,diferente,0.985,FABIANU SANTUS SIUVA,FABIANU SANTUS SIUVA,MARIA LURDES DULTRA SANTOS,MARIA LOURDES DUTRA SANTOS,2009-01-01,2009-01-10
2,0.385539,parecidas,exato,diferente,0.935,LUANA PEREIRA SIUVA,LUANA PEREIRA SIUVA,CLAUDIA PEREIRA SILVA,LUCIANA PEREIRA SILVA,2005-09-07,2001-01-07
3,0.385539,parecidas,exato,diferente,0.944,JOAU PAULU SOUZA SIUVA,JOAU PAULU SOUZA SIUVA,CARLA REGINA SILVA SOUZA,KARLA REGINA SILVA SOUSA,2003-11-09,2001-12-09
4,0.382587,parecidas,exato,uma_nula,0.925,ELIZANJELA VIEIRA SIUVA,ELIZANJELA VIEIRA SIUVA,DORACIR VIEIRA SILVA,DORACY VIEIRA SILVA,None,1979-08-01
5,0.382587,parecidas,exato,uma_nula,0.982,MARIA EDUARDA MORAIS SOUZA,MARIA EDUARDA MORAIS SOUZA,VERIDIANA MORAES SOUSA,VERIDIANA MORAES SOUZA,None,2006-07-02
6,0.382587,parecidas,exato,uma_nula,0.977,ANA PAULA FEREIRA SANTUS,ANA PAULA FEREIRA SANTUS,FRANCILENE FEREEIRA SANTOS,FRANCILENE FERREIRA SANTOS,None,2001-06-25
7,0.382072,parecidas,exato,diferente,0.972,RIAN SOUZA SANTUS,RIAN SOUZA SANTUS,FRACILENE GUIMARAES SOUSA,FRANCILENE GUIMARAES SOUSA,2001-12-21,2001-12-22
8,0.373713,parecidas,exato,uma_nula,0.943,LUANA SANTUS KOSTA,LUANA SANTUS KOSTA,MARIA COCEICAO SANTOS,MARIA CONCEICAO SANTOS,None,2000-10-19
9,0.373713,parecidas,exato,uma_nula,0.947,DOUGUILAS ARAUJU SIUVA,DOUGUILAS ARAUJU SIUVA,FRACIONETE OLVEIRA ARAUJO,GRACIONETE OLIVEIRA ARAUJO,None,2013-06-13


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.984569,sem_palavra,exato,diferente,0.683,JOZI MORAIS KARNEIRU,JOZI MORAIS KARNEIRU,MARIA JOSE PEREIRA,ELIETH MORAIS CARNEIRO,1980-08-26,1990-08-26
1,0.984569,sem_palavra,exato,diferente,0.656,RAIMUNDU AKINU PINHEIRU JUNIOR,RAIMUNDU AKINU PINHEIRU JUNIOR,LUCIMEIRE BARROS PINHEIRO,LUCILENE CONCEICAO MORAES,2021-12-10,2001-12-10
2,0.984569,sem_palavra,exato,diferente,0.497,MARIA LUZIA LARANJEIRA SANTUS,MARIA LUZIA LARANJEIRA SANTUS,GERTRUDIZ AMORM,MARIA SOCORRO LARANJEIRA,1998-08-15,1988-08-15
3,0.984569,sem_palavra,exato,diferente,0.508,JOZI EDIVAN ROXA GOMIS,JOZI EDIVAN ROXA GOMIS,RAIMUNDA PEREIRA SANTOS,JOANA ROCHA GOMES,1971-08-30,1975-08-30
4,0.984569,sem_palavra,exato,diferente,0.593,MARIA ELENA LINDOZU GAUVAU,MARIA ELENA LINDOZU GAUVAU,SONIA MARIA SILVA RIBEIRO,FELOMENA LINDOSO GALVAO,1984-05-25,1982-05-25
5,0.984569,sem_palavra,exato,diferente,0.545,MARIA GUIRASIELI SOUZA LIMA,MARIA GUIRASIELI SOUZA LIMA,MARTHA ARIELE RODRIGUEES SANTOS SOUSA,GLEYSON LIMA SILVA,2016-12-26,2014-12-26
6,0.984569,sem_palavra,exato,diferente,0.653,RAIMUNDU GUSTAVU SANTUS SOUZA,RAIMUNDU GUSTAVU SANTUS SOUZA,ANA ELIZA,GIORLANA SANTOS,2017-04-13,2014-04-13
7,0.984569,sem_palavra,exato,diferente,0.601,MARIA EMANUELI MENDIS AUVIS,MARIA EMANUELI MENDIS AUVIS,JOSILEIA MORENA,LUZILEA MORENO MENDES,2007-05-31,2017-05-31
8,0.984569,sem_palavra,exato,diferente,0.672,FRANSISKU PEDRU SIUVA AUVIS,FRANSISKU PEDRU SIUVA AUVIS,MARIA VIRGENS LOPES SOUSA,PATRICIA GOMES SILVA,2005-10-31,2008-10-31
9,0.984569,sem_palavra,exato,diferente,0.652,ANA VITORIA VIEIRA KONSEIKAU,ANA VITORIA VIEIRA KONSEIKAU,JULIANA CONCEICAO MARTINS OLIVEIRA,LEIDIANE VIEIRA SILVA,2021-08-30,2012-08-30


In [6]:
con.close()
